In [1]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings("ignore")
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from arch import arch_model


def rmse(a, p):
    return np.sqrt(np.mean((a - p) ** 2))


SECTORS    = ["XLF", "XLI", "XLV", "XLY", "XLK", "XLE"]
HOR        = {"h1": 1, "h5": 5, "h22": 22}
TEST_YEARS = [2020, 2021, 2022, 2023, 2024]
HAR  = ["RV_daily", "RV_weekly", "RV_monthly"]
HARX = HAR + ["CDS_level", "CDS_change_5d", "VIX", "Yield_slope", "BAA10Y"]

# Sector composite dataset. Six firm Energy composite; EXE excluded.
df = pd.read_parquet("Sector_Composites.parquet")

# Forward-mean targets at each horizon, per sector.
frames = []
for s in SECTORS:
    sub = df[df["Sector"] == s].sort_index().copy()
    for name, H in HOR.items():
        sub[f"Target_{name}"] = sub["RV_daily"].rolling(H, min_periods=H).mean().shift(-H)
    frames.append(sub)
big = pd.concat(frames)

In [2]:
import openpyxl

wb = openpyxl.load_workbook("DISSERTATION_FULL_DATA_FINAL.xlsx", read_only=True)
ws = wb["ETF PRICE DATA"]
rows = [r[:8] for r in ws.iter_rows(values_only=True)]
wb.close()

hdr = [str(x).strip() for x in rows[0]]
ETF_MAP = {"Financials": "XLF", "Energy": "XLE", "Technology": "XLK",
           "Healthcare": "XLV", "Industrials": "XLI", "Consumer Discretionary": "XLY"}
colidx = {ETF_MAP[h]: i for i, h in enumerate(hdr) if h in ETF_MAP}

dates, pdata = [], {s: [] for s in SECTORS}
for row in rows[2:]:
    try:
        dt = pd.to_datetime(row[0])
    except (ValueError, TypeError):
        continue
    if pd.isna(dt):
        continue
    dates.append(dt)
    for s in SECTORS:
        v = row[colidx[s]] if s in colidx else None
        pdata[s].append(pd.to_numeric(v, errors="coerce") if v is not None else np.nan)

price   = pd.DataFrame(pdata, index=pd.DatetimeIndex(dates)).sort_index()
returns = np.log(price).diff().dropna()

In [3]:
RF_SECTOR  = dict(n_estimators=300, max_features=0.7, random_state=42, n_jobs=-1)
XGB_SECTOR = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42, verbosity=0)

preds, peryear = {}, {}
for hz, H in HOR.items():
    tgt = f"Target_{hz}"
    for s in SECTORS:
        sub = big[big["Sector"] == s].dropna(subset=HARX + [tgt]).sort_index()
        r   = returns[s].dropna()
        acc = {k: [] for k in ["actual", "HAR-RV", "HAR-RV-X", "GJR-GARCH", "RF", "XGBoost"]}
        py  = {}
        for ty in TEST_YEARS:
            tr = sub[sub.index.year <  ty]
            te = sub[sub.index.year == ty]
            if len(te) == 0 or len(tr) < 200:
                continue
            y, a = tr[tgt].values, te[tgt].values
            acc["actual"].append(a)
            ph = np.clip(LinearRegression().fit(tr[HAR].values,  y).predict(te[HAR].values),  0, None)
            px = np.clip(LinearRegression().fit(tr[HARX].values, y).predict(te[HARX].values), 0, None)
            pr = np.clip(RandomForestRegressor(**RF_SECTOR).fit(tr[HARX].values, y).predict(te[HARX].values), 0, None)
            pg = np.clip(xgb.XGBRegressor(**XGB_SECTOR).fit(tr[HARX].values, y).predict(te[HARX].values), 0, None)
            # GJR-GARCH(1,1,1) on sector ETF returns. Estimated on the training window, applied to the test year. 
            # Returns scaled by 100 for numerical stability, rescaled to annualised-variance units.
            r_tr = r[r.index.year <  ty]
            res  = arch_model(r_tr * 100, mean="Zero", vol="Garch", p=1, o=1, q=1, dist="Normal").fit(disp="off", show_warning=False)
            r_up = r[r.index.year <= ty]
            fx   = arch_model(r_up * 100, mean="Zero", vol="Garch", p=1, o=1, q=1, dist="Normal").fix(res.params)
            cv   = ((fx.conditional_volatility / 100) ** 2 * 252).rolling(H, min_periods=H).mean()
            pgar = cv.reindex(te.index).values
            for m, p in [("HAR-RV", ph), ("HAR-RV-X", px), ("GJR-GARCH", pgar), ("RF", pr), ("XGBoost", pg)]:
                acc[m].append(p)
            vmask = ~np.isnan(pgar)
            py[ty] = {"HAR-RV": rmse(a, ph), "HAR-RV-X": rmse(a, px),
                      "GJR-GARCH": rmse(a[vmask], pgar[vmask]),
                      "RF": rmse(a, pr), "XGBoost": rmse(a, pg)}
        preds[(hz, s)]   = {m: np.concatenate(acc[m]) for m in acc}
        peryear[(hz, s)] = py
    print(f"{hz} done")

h1 done
h5 done
h22 done


In [4]:
MODELS = ["HAR-RV", "HAR-RV-X", "GJR-GARCH", "RF", "XGBoost"]
rows_out = []
for hz in HOR:
    for s in SECTORS:
        p = preds[(hz, s)]
        a = p["actual"]
        vals = {}
        for m in MODELS:
            if m == "GJR-GARCH":
                mask = ~np.isnan(p[m])
                vals[m] = rmse(a[mask], p[m][mask])
            else:
                vals[m] = rmse(a, p[m])
        rows_out.append({"Horizon": hz, "Sector": s, **{m: round(vals[m], 4) for m in MODELS}})

sector_rmse = pd.DataFrame(rows_out)
sector_rmse.to_csv("Sector_GARCH_RMSE.csv", index=False)

# One-day horizon, sector composite.
print("Sector-composite RMSE, one-day horizon:\n")
h1 = sector_rmse[sector_rmse["Horizon"] == "h1"].set_index("Sector")
for s in SECTORS:
    row = h1.loc[s]
    best = min(MODELS, key=lambda m: row[m])
    print(f"{s:<6}" + "".join(f"{row[m]:>10.4f}" + ("*" if m == best else " ") for m in MODELS))

Sector-composite RMSE, one-day horizon:

XLF       0.2388     0.2418     0.2380*    0.2557     0.2604 
XLI       0.1829     0.1892     0.1741*    0.1960     0.2042 
XLV       0.1084     0.1126     0.1075*    0.1225     0.1266 
XLY       0.1884     0.1896     0.1859*    0.1914     0.2008 
XLK       0.2223*    0.2304     0.2228     0.2446     0.2561 
XLE       0.4923     0.5030     0.4893*    0.5314     0.5311 
